# Get to Know a Dataset: ARPA-I INSIGHTS

This notebook serves as a guided tour of the [ARPA-I INSIGHTS](https://registry.opendata.aws/arpa-i-insights/) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

## About the Dataset

The ARPA-I INSIGHTS Dataset is a large-scale, high-density (~85 pts/m²) geiger-mode aerial LiDAR dataset covering over 1,600 km² across the Salt Lake City, UT and Denver, CO metropolitan regions, including major freeway corridors. Collected in June 2025, the dataset provides dense three-dimensional measurements with approximately 30 cm vertical and horizontal point spacing, enabling detailed representation of transportation infrastructure and surrounding urban environments.

The data was acquired as part of a U.S. Department of Transportation ARPA-I-funded project, executed by MIT Lincoln Laboratory, to support state and regional transportation agencies. The dataset is designed to enable transportation digital twins, supporting virtual inspection, measurement, and inventory of infrastructure assets at corridor and regional scales.

The data is optimized for cloud-based access through tiled storage, STAC catalog with GeoParquet index, and analysis-ready derivatives, supporting scalable workflows for geospatial analysis, infrastructure monitoring, and multimodal AI research.

**License:** CC-BY-4.0 (Data), MIT License (Code)

**Maintainers:** MIT Lincoln Laboratory

**Sponsors:** U.S. Department of Transportation ARPA-I

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The ARPA-I INSIGHTS dataset is organized in an S3 bucket with the following structure:

```
s3://arpa-i-insights/
├── lidar/
│   ├── data/                # L3 LAS files, one folder per sortie
│   │   ├── DRCOG/           # Denver Regional Council of Governments
│   │   ├── SLC/             # Salt Lake City
│   │   ├── I15South/        # Interstate 15 South
│   │   ├── I25N/            # Interstate 25 North
│   │   ├── I70*             # Various Interstate 70 segments
│   │   ├── I80*             # Various Interstate 80 segments
│   │   └── FrontRange/      # Front Range corridor
│   └── stac/
│       └── index/
│       │   └── items.parquet  # STAC catalog index
│       └── items/           # STAC items (.json), one folder per sortie   
│       │   └── DRCOG/       # STAC items (.json) for DRCOG
│       │   └── SLC/         # STAC items (.json) for SLC
│       │   └── ...          # continue for other sorties
│       └── collection.json  # STAC collection json
└── [additional derivatives and annotations to come]
```

Each sortie (data collection flight) is organized as a separate prefix under `lidar/data/`. The point cloud data is stored as tiled LAS files for efficient access and processing. The STAC (SpatioTemporal Asset Catalog) index provides a queryable catalog of all available tiles with spatial, temporal, and statistical metadata. A full STAC collection is available under `lidar/stac/collection.json` with respective items under `lidar/stac/items/`. A geoparquet index is provided in `lidar/stac/index/items.parquet` for fast spatial querying.

**Key Features:**
- **82,429 LAS tiles** covering over 1,600 km²
- **~140 billion points** total across all tiles
- **Tiled organization** for cloud-optimized access
- **STAC catalog** with GeoParquet index for fast spatial queries
- **Two coordinate systems:** NAD83(2011) / Colorado Central (EPSG:6430) and NAD83(2011) / Utah Central (EPSG:6626)

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following libraries:
# boto3 >= 1.38.23
# pandas >= 2.0.0
# pyarrow >= 10.0.0
# matplotlib >= 3.5.0
# laspy >= 2.4.0  (for working with LAS files)
# geopandas >= 0.12.0  (optional, for spatial operations)

# Import the libraries required for this notebook
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from botocore import UNSIGNED
from botocore.config import Config
import json
from io import BytesIO

Next, we will define the location of our dataset and create our boto3 S3 client. Because this is a public bucket, we don't need to sign requests.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "arpa-i-insights"

# Create S3 client for public bucket (no credentials required)
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# List the top-level prefixes
response = s3.list_objects_v2(Bucket=bucket, Delimiter='/', MaxKeys=10)
print("Top-level prefixes:")
for item in response.get('CommonPrefixes', []):
    print(f"  {item['Prefix']}")

Let's explore the structure of the lidar data directory to see the different sortie collections:

In [ ]:
# List the sorties (data collection flights) available
response = s3.list_objects_v2(Bucket=bucket, Prefix='lidar/data/', Delimiter='/', MaxKeys=50)
print("Available sorties:")
for item in response.get('CommonPrefixes', []):
    sortie = item['Prefix'].split('/')[-2]
    print(f"  {sortie}")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The ARPA-I INSIGHTS dataset uses two primary formats:

#### 1. LAS/LAZ Format (Point Cloud Data)

**LAS** (LASer) is the industry-standard binary format for storing airborne and terrestrial LiDAR point cloud data. **LAZ** is the compressed version using lossless compression. The format stores:

- **3D coordinates** (X, Y, Z) for each point
- **Intensity values** (return signal strength)
- **Return number** (first, second, last, etc.)
- **Classification codes** (ground, vegetation, building, etc.)
- **RGB color** (when available)
- **GPS timestamps**
- **Scan angle and point source ID**

**Why LAS?**
- Industry standard for LiDAR data exchange
- Compact and efficient storage
- Preserves all sensor metadata
- Widely supported by GIS and point cloud processing software

**Working with LAS files:**
- **Python:** Use [laspy](https://laspy.readthedocs.io/) for reading/writing LAS files
- **Command-line:** Use [PDAL](https://pdal.io/) for processing pipelines
- **Desktop GIS:** QGIS, ArcGIS Pro, Global Mapper
- **Point cloud tools:** CloudCompare, Pix4D, LAStools

#### 2. GeoParquet Format (STAC Index)

**GeoParquet** extends Apache Parquet with geospatial metadata, providing:
- Fast columnar queries
- Spatial filtering capabilities
- Efficient storage and transfer
- Schema evolution support

Our STAC index (`items.parquet`) contains metadata for all 82,429 tiles including:
- Spatial extents (bounding boxes)
- Point counts and density
- Elevation statistics
- File locations (S3 URIs and HTTPS URLs)
- Coordinate reference system information

**Working with the STAC index:**
- **Python:** Use pandas with pyarrow backend
- **Spatial queries:** Use GeoPandas for GIS operations
- **Cloud queries:** Amazon Athena, DuckDB

#### AWS Services for This Dataset

- **Amazon S3:** Direct access to LAS files via HTTPS or S3 API

### Q: Can you show us an example of downloading and loading the STAC catalog?

The STAC catalog provides a comprehensive index of all available LiDAR tiles. Let's load it and explore the metadata.

In [ ]:
# Download the STAC index from S3
stac_key = "lidar/index/stac/items.parquet"

print(f"Downloading STAC catalog from s3://{bucket}/{stac_key}")
obj = s3.get_object(Bucket=bucket, Key=stac_key)
stac_df = pd.read_parquet(BytesIO(obj['Body'].read()))

print(f"\nLoaded {len(stac_df):,} STAC items")
print(f"Columns: {list(stac_df.columns)}")

Let's examine the first few records to understand the structure:

In [ ]:
# Display first 5 records with key columns
display_cols = ['id', 'sortie', 'tile_id', 'pc:count', 'elevation:mean', 'datetime', 's3_path']
stac_df[display_cols].head()

Now let's look at summary statistics across the entire catalog:

In [ ]:
print("=== Dataset Summary ===")
print(f"Total tiles: {len(stac_df):,}")
print(f"Total points: {stac_df['pc:count'].sum():,.0f}")
print(f"\nSorties (collection areas): {stac_df['sortie'].nunique()}")
print(f"Coordinate systems: {stac_df['proj:epsg'].nunique()}")
print(f"Collection date: {stac_df['datetime'].iloc[0]}")

print("\n=== Point Count Statistics ===")
print(stac_df['pc:count'].describe())

print("\n=== Elevation Statistics (feet) ===")
print(stac_df[['elevation:min', 'elevation:max', 'elevation:mean']].describe())

### Q: A picture is worth a thousand words. Show us visualizations that illustrate the dataset.

Let's create several visualizations to understand the dataset's characteristics.

#### Distribution of Tiles Across Sorties

First, let's see how the data is distributed across different collection areas:

In [ ]:
# Count tiles per sortie
sortie_counts = stac_df['sortie'].value_counts().sort_values(ascending=True)

# Create horizontal bar chart
plt.figure(figsize=(12, 8), dpi=100, facecolor='white')
plt.barh(sortie_counts.index, sortie_counts.values, color='#2E86AB', edgecolor='white', linewidth=0.5)

plt.title('Distribution of LiDAR Tiles Across Collection Areas', fontsize=16, pad=20, fontweight='bold')
plt.xlabel('Number of Tiles', fontsize=12, labelpad=10)
plt.ylabel('Sortie / Collection Area', fontsize=12, labelpad=10)
plt.grid(True, axis='x', linestyle='--', alpha=0.3, color='gray')

ax = plt.gca()
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

plt.tight_layout()
plt.show()

print(f"\nLargest collection: {sortie_counts.index[-1]} with {sortie_counts.values[-1]:,} tiles")
print(f"Total collection areas: {len(sortie_counts)}")

#### Point Density Distribution

Let's examine the distribution of point counts per tile:

In [ ]:
plt.figure(figsize=(12, 7), dpi=100, facecolor='white')

plt.hist(stac_df['pc:count'], 
         bins=50,
         color='#A23B72',
         edgecolor='white',
         linewidth=1.2,
         alpha=0.8)

plt.title('Distribution of Point Counts per Tile', 
         fontsize=16, 
         pad=20, 
         fontweight='bold')
plt.xlabel('Number of Points', fontsize=12, labelpad=10)
plt.ylabel('Number of Tiles', fontsize=12, labelpad=10)

# Add vertical line for mean
mean_points = stac_df['pc:count'].mean()
plt.axvline(mean_points, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_points:,.0f}')
plt.legend()

plt.grid(True, linestyle='--', alpha=0.3, color='gray')

ax = plt.gca()
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

plt.tight_layout()
plt.show()

#### Elevation Distribution

The dataset covers significant elevation changes from urban valleys to mountain corridors:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100, facecolor='white')

# Elevation mean histogram
axes[0].hist(stac_df['elevation:mean'], 
             bins=50,
             color='#18A558',
             edgecolor='white',
             linewidth=1.2,
             alpha=0.8)
axes[0].set_title('Mean Elevation Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Mean Elevation (feet)', fontsize=12)
axes[0].set_ylabel('Number of Tiles', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.3)
axes[0].set_facecolor('#f8f9fa')

# Elevation range (max - min) per tile
elevation_range = stac_df['elevation:max'] - stac_df['elevation:min']
axes[1].hist(elevation_range, 
             bins=50,
             color='#F18F01',
             edgecolor='white',
             linewidth=1.2,
             alpha=0.8)
axes[1].set_title('Elevation Range per Tile', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Elevation Range (feet)', fontsize=12)
axes[1].set_ylabel('Number of Tiles', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.3)
axes[1].set_facecolor('#f8f9fa')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

plt.tight_layout()
plt.show()

print(f"Elevation range across dataset: {stac_df['elevation:min'].min():.1f} - {stac_df['elevation:max'].max():.1f} feet")
print(f"Mean elevation: {stac_df['elevation:mean'].mean():.1f} feet")

### Q: How can I process LiDAR data from this dataset?

Let's demonstrate a typical workflow for downloading and analyzing a single LAS file. This example shows how to:

1. Select a tile from the STAC catalog
2. Download the LAS file
3. Read and analyze the point cloud data
4. Extract basic statistics

**Note:** For production workflows with many tiles, consider using PDAL pipelines or parallel processing with AWS Batch.

In [ ]:
# Select a tile with a reasonable number of points for demo
sample_tile = stac_df[
    (stac_df['pc:count'] > 1000000) & 
    (stac_df['pc:count'] < 2000000)
].sample(1).iloc[0]

print("Selected tile for analysis:")
print(f"  ID: {sample_tile['id']}")
print(f"  Sortie: {sample_tile['sortie']}")
print(f"  Point count: {sample_tile['pc:count']:,}")
print(f"  Elevation range: {sample_tile['elevation:min']:.1f} - {sample_tile['elevation:max']:.1f} ft")
print(f"  S3 path: {sample_tile['s3_path']}")

Now let's download and read the LAS file using laspy:

In [ ]:
import laspy

# Extract S3 key from full path
s3_path = sample_tile['s3_path']
s3_key = s3_path.replace(f's3://{bucket}/', '')

print(f"Downloading {s3_key}...")
las_obj = s3.get_object(Bucket=bucket, Key=s3_key)
las_data = las_obj['Body'].read()

# Read LAS file
las = laspy.read(BytesIO(las_data))

print(f"\nLAS File Header Information:")
print(f"  Point format: {las.point_format.id}")
print(f"  Point count: {len(las.points):,}")
print(f"  Number of returns: {las.header.number_of_points_by_return}")
print(f"  Scale factors: {las.header.scales}")
print(f"  Offsets: {las.header.offsets}")

# Access point data
x = las.x
y = las.y
z = las.z
intensity = las.intensity
return_num = las.return_number
classification = las.classification

print(f"\nPoint Data Statistics:")
print(f"  X range: {x.min():.2f} - {x.max():.2f}")
print(f"  Y range: {y.min():.2f} - {y.max():.2f}")
print(f"  Z range: {z.min():.2f} - {z.max():.2f}")
print(f"  Intensity range: {intensity.min()} - {intensity.max()}")

Let's visualize the elevation distribution and return numbers:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100, facecolor='white')

# Elevation histogram
axes[0].hist(z, bins=100, color='#2E86AB', alpha=0.7, edgecolor='white')
axes[0].set_title('Elevation Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Elevation', fontsize=12)
axes[0].set_ylabel('Number of Points', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.3)
axes[0].set_facecolor('#f8f9fa')

# Return number distribution
return_counts = np.bincount(return_num)
axes[1].bar(range(len(return_counts)), return_counts, color='#A23B72', alpha=0.7)
axes[1].set_title('Return Number Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Return Number', fontsize=12)
axes[1].set_ylabel('Number of Points', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.3, axis='y')
axes[1].set_facecolor('#f8f9fa')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nReturn distribution:")
for i, count in enumerate(return_counts):
    if count > 0:
        pct = 100 * count / len(las.points)
        print(f"  Return {i}: {count:,} points ({pct:.1f}%)")

### Visual Analysis with GUI Tools

While programmatic analysis is powerful for large-scale processing, GUI-based point cloud viewers are invaluable for:

- **Visual exploration and quality assessment** of point cloud data
- **Manual annotation and labeling** for machine learning training data
- **Interactive measurement** of distances, areas, and volumes
- **Point cloud cleanup** (noise removal, outlier detection)
- **3D visualization** with various rendering modes (elevation, intensity, classification, RGB)
- **Cross-section analysis** for infrastructure inspection

#### Recommended GUI Tools

**CloudCompare** (Free, Open Source)
- Download: [https://www.cloudcompare.org/](https://www.cloudcompare.org/)
- Excellent for general point cloud visualization and analysis
- Supports classification coloring, scalar fields, and cross-sections
- Built-in tools for segmentation, measurement, and comparison

**QGIS with Point Cloud Support** (Free, Open Source)
- 2D/3D visualization integrated with GIS workflows
- Layer-based approach for combining point clouds with vector/raster data

**Other Options:**
- **LAStools/LASview** - Fast viewer for LAS files
- **Pix4Dmatic** - Commercial software with advanced features
- **ArcGIS Pro** - Enterprise GIS with LAS support
- **Global Mapper** - Commercial tool with terrain analysis

#### Example: Loading Data in CloudCompare

After downloading a LAS file (as shown above), you can open it in CloudCompare:

1. **Open the file:** File → Open → Select your downloaded .las file
2. **Apply coloring:** Use scalar fields to color by elevation, intensity, classification, or return number
3. **Create cross-sections:** Tools → Segmentation → Extract Sections
4. **Measure features:** Tools → Point Picking → Point-to-point distance
5. **Export views:** Capture screenshots or export processed data

Below is an example of what a LiDAR tile from the ARPA-I INSIGHTS dataset looks like when loaded in CloudCompare:

![CloudCompare visualization example](cloudcompare_example.png)

*Example visualization of a tile colored by elevation. The high point density (~85 pts/m²) enables detailed representation of road surfaces, overpasses, barriers, signage, vegetation, and surrounding terrain. CloudCompare's scalar field coloring helps identify elevation changes, road grades, and vertical infrastructure elements.*

**What you can see in typical tiles:**
- Road surfaces with lane-level detail
- Overhead structures (bridges, overpasses, gantries)
- Roadside infrastructure (guardrails, barriers, signs, poles)
- Vegetation (trees, shrubs, grass)
- Adjacent buildings and parking areas
- Terrain elevation changes and slopes

#### Typical Visualization Workflows

**Infrastructure Inspection:**
- Color by elevation to identify road surface and grade changes
- Use cross-sections to measure clearances and slopes
- Filter by classification to isolate specific features

**Training Data Annotation:**
- Manually segment and label features of interest
- Export labeled subsets for machine learning
- Validate automated classification results

**Quality Control:**
- Identify data gaps or low-density areas
- Detect noise points and outliers
- Verify georeferencing accuracy with known control points

**Tip:** For large tiles with millions of points, CloudCompare may require significant RAM. Consider using the point cloud subsampling feature or processing smaller areas of interest.

### Advanced Analysis Example: Ground Classification and DEM Generation

For more advanced analysis workflows, the dataset can be processed using PDAL (Point Data Abstraction Library) to:

1. **Classify ground points** using algorithms like SMRF (Simple Morphological Filter) or PMF (Progressive Morphological Filter)
2. **Generate Digital Elevation Models (DEMs)** from ground-classified points
3. **Compute terrain derivatives** such as slope and roughness

A complete example script is available in the repository at `examples/scripts/analyze_las.py`. This script demonstrates:

- Reading LAS files and classifying ground points
- Generating DEMs at user-specified resolutions (e.g., 1 ft, 30 cm, 1 m)
- Computing slope and roughness rasters
- Parallel processing of multiple tiles

**Example workflow:**

```bash
# First, download LAS files from S3 to local storage
# (The analyze_las.py script requires local filesystem paths)
aws s3 sync s3://arpa-i-insights/lidar/data/DRCOG/ ./local_data/DRCOG/ \
    --no-sign-request

# Install PDAL and dependencies
conda install -c conda-forge pdal python-pdal rasterio scipy

# Run the analysis script on local data
python scripts/data/analysis/analyze_las.py \
    --in ./local_data/DRCOG/ \
    --out ./output \
    --resolution 1m \
    --roughness-window 3m \
    --workers 8
```

**Output structure:**
```
output/
├── DEM/
│   └── 1m/              # Digital elevation models
├── L4_las/              # Ground-classified LAS files
├── roughness/
│   └── 1m/              # Surface roughness rasters
└── slope/
    └── 1m/              # Slope rasters (percent or degrees)
```

This workflow is particularly useful for:
- Infrastructure monitoring (road surface analysis)
- Vegetation management (clearance analysis)
- Flood modeling (terrain analysis)
- Asset inventory (pole/sign detection)

**Note:** For cloud-native workflows that process data directly from S3 without downloading, consider using PDAL's [readers.ept](https://pdal.io/en/stable/stages/readers.ept.html) or custom AWS Lambda functions with streaming LAS readers.

### Q: What are some unanswered questions that could be explored with this dataset?

The ARPA-I INSIGHTS dataset opens up numerous research and application opportunities:

#### Infrastructure Digital Twins
- **Virtual inspection workflows:** How can transportation agencies use dense LiDAR to replace or augment physical roadway inspections?
- **Change detection:** Can we automatically detect infrastructure changes, damage, or maintenance needs by comparing multi-temporal collections?
- **Asset inventory:** How accurately can we automatically inventory signs, signals, poles, barriers, and other roadside assets?

#### Machine Learning Applications
- **Semantic segmentation:** Can we train models to classify surfaces (road, sidewalk, trail) and objects (poles, signs, vehicles) at scale?
- **Feature extraction:** What automated methods work best for extracting lane markings, curbs, and road edges from dense LiDAR?
- **Transfer learning:** How well do models trained on one metropolitan area transfer to another?

#### Transportation Planning
- **Multimodal network analysis:** How can we use LiDAR to comprehensively map pedestrian, bicycle, and vehicle infrastructure?
- **Accessibility assessment:** Can we automatically identify ADA compliance issues like missing curb ramps or excessive slopes?
- **Safety analysis:** How does roadside infrastructure geometry correlate with crash data?

#### Environmental Analysis
- **Urban heat island effects:** Can we correlate surface materials (detected via LiDAR) with temperature data?
- **Stormwater management:** How can high-resolution terrain models improve drainage network analysis?
- **Vegetation encroachment:** Can we automate detection of sight distance obstructions?

#### Methodological Challenges
- **Cloud processing:** What are the most efficient architectures for processing 140+ billion points in the cloud?
- **Spatial indexing:** How can we optimize spatial queries across tiled datasets at this scale?
- **Data fusion:** How do we best combine LiDAR with imagery, mobile mapping, and other data sources?

**Getting Started:**
- Start with a single sortie (e.g., a highway corridor) to develop and test methods
- Use the STAC catalog for spatial queries to find relevant tiles
- Leverage AWS services (Batch, Lambda, SageMaker) for scalable processing
- Consider the provided analysis scripts as starting points for custom workflows

We encourage the community to share findings, tools, and applications developed using this dataset!

---

## License and Citation

**Data License:** [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/)

**Code License:** [MIT License](https://opensource.org/licenses/MIT)

**Maintainers:** MIT Lincoln Laboratory

**Sponsors:** U.S. Department of Transportation Advanced Research Projects Agency - Infrastructure (ARPA-I)

**Citation:**
```
ARPA-I INSIGHTS LiDAR Dataset (2026). MIT Lincoln Laboratory. 
U.S. Department of Transportation ARPA-I. 
Available at: https://registry.opendata.aws/arpa-i-insights/
```

**Contact:** For questions or support, please visit the [GitHub repository](https://github.com/mit-ll/USDOT-INSIGHTS) or the [Registry of Open Data on AWS](https://registry.opendata.aws/arpa-i-insights/).